Define the raw dataset
- BOM: raw materials required to produce each finished product (incl. standard quantity + loss rate)
- ITEM: Classify item codes by group: tools, finished products, outsource, screws or consumables used in the workshop, origin.
- TRACK_PO: list of item codes and PO No with the information: PO quantity, Received Qty, Remain, PO status, Total Amount
- SO: list of item codes and SO with Order Qty, Delivery Qty, Delivery Date, SO status
- INVENTORY: shows the quantity of each item codes in all the warehouses


# I. IMPORT THE DATA

## 1. Import the necessary package

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import openpyxl
import statsmodels.api as sm

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"


pd.options.display.float_format = '{:,.4f}'.format

## 2. Import the input data

In [2]:
# Add the working directory to the system path
path       = "/Users/vinc-tere/Library/CloudStorage/OneDrive-Personal/Documents/0. Cuong/Data Analyst/Project/OHTA"

# Import the raw dataset: BOM, ITEM, PURCHASE ORDER, INVENTORY, SALE PRICE
list_sheet = ["bom", "item", "track_po", "inventory", "sale_price", "so"]

for sheet in list_sheet:
    globals()[f"df_{sheet}"] = pd.read_excel(os.path.join(path, "OHTA DATA.xlsx"), sheet_name= sheet)


# II. DATA PROCESSING

This step aims to understand the data structure to format data into suitable data and clean the data 

#### 1. Inventory

In [3]:
df_inventory.head() # Display the first few rows of the inventory

,Item Group,Item Category,Sub Category,Item Code,Unit,Warehouse,Minimum Stock,Reservation,Available,OK,QA Waiting,Remade,NG,Total
0,Tools,Tool Others,Tool Others,0.25MM GF WIDE,Kg,Tools,NaN,0,40.0000,40.0000,0,0.0000,0,40.0000
1,Screws,Screw Others,Screw Others,00-01-J070-00500310-05-002,Pcs,Screws,NaN,0,300.0000,300.0000,0,0.0000,0,300.0000
2,Screws,Tools,Tools,001-3/16,Pcs,Screws,NaN,0,1.0000,1.0000,0,0.0000,0,1.0000
3,Internal Finish Products,Internal Finish Products,Internal Finish Products,002-50-021,Pcs,Products,0.0000,0,1.0000,1.0000,0,0.0000,0,1.0000
4,Internal Finish Products,Internal Finish Products,Internal Finish Products,002-50-022,Pcs,Products,0.0000,0,1.0000,1.0000,0,0.0000,0,1.0000


In [4]:
df_inventory.info() # Get information about the inventory DataFrame, including data types and non-null counts

<class 'pandas.DataFrame'>
RangeIndex: 8676 entries, 0 to 8675
Data columns (total 14 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Item Group     8676 non-null   str    
 1   Item Category  8676 non-null   str    
 2   Sub Category   8676 non-null   str    
 3   Item Code      8675 non-null   object 
 4   Unit           8676 non-null   str    
 5   Warehouse      8676 non-null   str    
 6   Minimum Stock  5292 non-null   float64
 7   Reservation    8676 non-null   int64  
 8   Available      8676 non-null   float64
 9   OK             8676 non-null   float64
 10  QA Waiting     8676 non-null   int64  
 11  Remade         8676 non-null   float64
 12  NG             8676 non-null   int64  
 13  Total          8676 non-null   float64
dtypes: float64(5), int64(3), object(1), str(5)
memory usage: 949.1+ KB


In [5]:
df_inventory.shape # Get the shape of the inventory DataFrame (number of rows and columns)

(8676, 14)

Check if the null is in the data

In [6]:
df_inventory.isnull().sum() # Check for missing values in the inventory DataFrame

Item Group          0
Item Category       0
Sub Category        0
Item Code           1
Unit                0
Warehouse           0
Minimum Stock    3384
Reservation         0
Available           0
OK                  0
QA Waiting          0
Remade              0
NG                  0
Total               0
dtype: int64

In [7]:
# Because only the minimum stock contains missing values, these cells will be filled with 0, which means that there is no minimum stock requirement for those items.
df_inventory["Minimum Stock"] =df_inventory["Minimum Stock"].fillna(0, inplace = False)

# After checking, the rows with "item code" null values will be dropped because they are not useful for the analysis.
df_inventory = df_inventory.dropna(subset=["Item Code"], inplace = False)

# Verify that the missing values in the "Minimum Stock" column have been filled with 0.
df_inventory.isnull().sum() 

Item Group       0
Item Category    0
Sub Category     0
Item Code        0
Unit             0
Warehouse        0
Minimum Stock    0
Reservation      0
Available        0
OK               0
QA Waiting       0
Remade           0
NG               0
Total            0
dtype: int64

Check the simple statistics

In [8]:
df_inventory.describe() # Get summary statistics of the inventory DataFrame

main_quantity = ["Reservation", "Available", "OK", "QA Waiting", "Remade", "NG", "Total"]

,Minimum Stock,Reservation,Available,OK,QA Waiting,Remade,NG,Total
count,"8,675.0000","8,675.0000","8,675.0000","8,675.0000","8,675.0000","8,675.0000","8,675.0000","8,675.0000"
mean,"1,715.3428",301.5292,"2,398.6294","2,700.1587",165.1266,3.9912,0.2071,"2,869.4836"
std,"24,494.1439","6,223.0988","24,794.8015","27,941.4774","10,289.6801",222.5408,11.9223,"29,997.3063"
min,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
25%,0.0000,0.0000,5.0000,5.0000,0.0000,0.0000,0.0000,6.0000
50%,0.0000,0.0000,100.0000,116.0000,0.0000,0.0000,0.0000,125.0000
75%,10.0000,0.0000,940.0000,970.0000,0.0000,0.0000,0.0000,980.5000
max,"1,570,000.0000","468,000.0000","1,396,800.0000","1,411,000.0000","940,000.0000","16,913.0000",970.0000,"1,411,000.0000"


In [9]:
df_item.head()

,Accounting Code,Item Code,Supplier Item Code,Maker Item Code,Item Name (VN),Item Name (EN),Item Name (JP),Maker,Manage Volume,Item Type,Item Group,Item Category,Jan Code,SPQ,Sub Category,Primary Unit,HS Code,Origin Of Goods,Weight (Gram),Status
0,NaN,#100,NaN,NaN,Giấy nhám KOVAX #100,emery paper KOVAX #100,emery paper KOVAX #100,NaN,No,Tools and Office Supplies,Tools,Tool Others,NaN,NaN,Tool Others,Sheet,NaN,Japan,NaN,Active
1,4001528,#1180-08,NaN,NaN,PETRATION GAGE,PETRATION GAGE,PETRATION GAGE,NaN,No,Tools and Office Supplies,Tools,Tool Others,NaN,NaN,Tool Others,Pcs,NaN,NaN,NaN,Active
2,4001529,#1184-08,NaN,NaN,GO-NO/GO GAUGE,GO-NO/GO GAUGE,GO-NO/GO GAUGE,NaN,No,Tools and Office Supplies,Tools,Tool Others,NaN,NaN,Tool Others,Pcs,NaN,NaN,NaN,Active
3,NaN,#120,NaN,NaN,Giấy nhám KOVAX #120,emery paper KOVAX #120,emery paper KOVAX #120,NaN,No,Tools and Office Supplies,Tools,Tool Others,NaN,NaN,Tool Others,Sheet,NaN,NaN,NaN,Active
4,3018337,#500SP1-SL101-D16+26+37,NaN,NaN,OILES Bush SPB φ16x26x37,OILES Bush SPB φ16x26x37,オイレス ブッシュ SPB φ16x26x37,OILES JP,No,Goods,Screws,Screw Others,NaN,10,Screw Others,Pcs,NaN,Japan,NaN,Active


### 2. Items

In [10]:
# Display the first few rows of the item DataFrame
df_item.head(5) 

,Accounting Code,Item Code,Supplier Item Code,Maker Item Code,Item Name (VN),Item Name (EN),Item Name (JP),Maker,Manage Volume,Item Type,Item Group,Item Category,Jan Code,SPQ,Sub Category,Primary Unit,HS Code,Origin Of Goods,Weight (Gram),Status
0,NaN,#100,NaN,NaN,Giấy nhám KOVAX #100,emery paper KOVAX #100,emery paper KOVAX #100,NaN,No,Tools and Office Supplies,Tools,Tool Others,NaN,NaN,Tool Others,Sheet,NaN,Japan,NaN,Active
1,4001528,#1180-08,NaN,NaN,PETRATION GAGE,PETRATION GAGE,PETRATION GAGE,NaN,No,Tools and Office Supplies,Tools,Tool Others,NaN,NaN,Tool Others,Pcs,NaN,NaN,NaN,Active
2,4001529,#1184-08,NaN,NaN,GO-NO/GO GAUGE,GO-NO/GO GAUGE,GO-NO/GO GAUGE,NaN,No,Tools and Office Supplies,Tools,Tool Others,NaN,NaN,Tool Others,Pcs,NaN,NaN,NaN,Active
3,NaN,#120,NaN,NaN,Giấy nhám KOVAX #120,emery paper KOVAX #120,emery paper KOVAX #120,NaN,No,Tools and Office Supplies,Tools,Tool Others,NaN,NaN,Tool Others,Sheet,NaN,NaN,NaN,Active
4,3018337,#500SP1-SL101-D16+26+37,NaN,NaN,OILES Bush SPB φ16x26x37,OILES Bush SPB φ16x26x37,オイレス ブッシュ SPB φ16x26x37,OILES JP,No,Goods,Screws,Screw Others,NaN,10,Screw Others,Pcs,NaN,Japan,NaN,Active


In [11]:
# Filtering the concerned items

# These item groups mainly contributes to the inventory and sales of the company, so they will be the focus of the analysis. The other item groups will be excluded because they are not relevant to the inventory and sales performance of the company.
item_group = [
                "Internal Finish Products",
                "External Finish Products", 
                "Services Processing Products",
                "OutSource Processing",
                "Screws"
                ]
item_type  = "Materials"

df_item_filtered = df_item[(df_item['Item Group'].isin(item_group)) | (df_item['Item Type'] == item_type)]
df_item_filtered.head() # Display the first few rows of the filtered item DataFrame

,Accounting Code,Item Code,Supplier Item Code,Maker Item Code,Item Name (VN),Item Name (EN),Item Name (JP),Maker,Manage Volume,Item Type,Item Group,Item Category,Jan Code,SPQ,Sub Category,Primary Unit,HS Code,Origin Of Goods,Weight (Gram),Status
4,3018337,#500SP1-SL101-D16+26+37,NaN,NaN,OILES Bush SPB φ16x26x37,OILES Bush SPB φ16x26x37,オイレス ブッシュ SPB φ16x26x37,OILES JP,No,Goods,Screws,Screw Others,NaN,10,Screw Others,Pcs,NaN,Japan,NaN,Active
5,2001319,#8310-01-01,NaN,NaN,φ20 Lens Holder,φ20 Lens Holder,20mmﾚﾝｽﾞﾎﾙﾀﾞ,NaN,No,Products,Internal Finish Products,Internal Finish Products,NaN,NaN,Internal Finish Products,Pcs,NaN,NaN,NaN,Active
6,2001320,#8382-02-01,NaN,NaN,φ20 Mirror Tube,φ20 Mirror Tube,φ20mm鏡筒,NaN,No,Products,Internal Finish Products,Internal Finish Products,NaN,NaN,Internal Finish Products,Pcs,NaN,NaN,NaN,Active
7,2001321,#8385-01-01,NaN,NaN,BSP Base,BSP Base,BSPﾍﾞｰｽ,NaN,No,Products,Internal Finish Products,Internal Finish Products,NaN,NaN,Internal Finish Products,Pcs,NaN,NaN,NaN,Active
8,2001322,#8386-01,NaN,NaN,Pedestal,Pedestal,台座,NaN,No,Products,Internal Finish Products,Internal Finish Products,NaN,NaN,Internal Finish Products,Pcs,NaN,NaN,NaN,Active


### 3. SO

In [12]:
df_so.head(4)
df_so.info()

,Customer Code,Customer Name,SO No,Item Code,Order Date,Order Qty,Delivery Qty,Delivery Date,Estimate Stock in Date,Issue Date,Remain Quantity,SO Status
0,C00001,"Ohta Co.,Ltd.",801104,M1TM1851-A01-02,2026-02-16,400,0,2026-08-26,2026-07-28,NaT,400,New
1,C00001,"Ohta Co.,Ltd.",798646,DF410842-A,2026-01-22,180,0,2026-06-27,2026-05-26,NaT,180,New
2,C00001,"Ohta Co.,Ltd.",798648,DF410848-02,2026-01-22,360,0,2026-06-27,2026-05-26,NaT,360,New
3,C00001,"Ohta Co.,Ltd.",798650,DF410847,2026-01-22,180,0,2026-06-27,2026-05-26,NaT,180,New


<class 'pandas.DataFrame'>
RangeIndex: 51563 entries, 0 to 51562
Data columns (total 12 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Customer Code           51563 non-null  str           
 1   Customer Name           51563 non-null  str           
 2   SO No                   51563 non-null  object        
 3   Item Code               51563 non-null  object        
 4   Order Date              51563 non-null  datetime64[us]
 5   Order Qty               51563 non-null  int64         
 6   Delivery Qty            51563 non-null  int64         
 7   Delivery Date           51563 non-null  datetime64[us]
 8   Estimate Stock in Date  12396 non-null  datetime64[us]
 9   Issue Date              50048 non-null  datetime64[us]
 10  Remain Quantity         51563 non-null  int64         
 11  SO Status               51563 non-null  str           
dtypes: datetime64[us](4), int64(3), object(2), str(3)
memory 

In [13]:
# Remove the Issue Date and Estimate Stock in Date
df_so = df_so.drop(columns=["Issue Date", "Estimate Stock in Date"])

# Extract the year and month from delivery date and create new columns for year and month
df_so['Delivery Year'] = df_so['Delivery Date'].dt.year
df_so['Delivery Month'] = df_so['Delivery Date'].dt.month

df_so.head() # Display the first few rows of the sales order DataFrame after processing

,Customer Code,Customer Name,SO No,Item Code,Order Date,Order Qty,Delivery Qty,Delivery Date,Remain Quantity,SO Status,Delivery Year,Delivery Month
0,C00001,"Ohta Co.,Ltd.",801104,M1TM1851-A01-02,2026-02-16,400,0,2026-08-26,400,New,2026,8
1,C00001,"Ohta Co.,Ltd.",798646,DF410842-A,2026-01-22,180,0,2026-06-27,180,New,2026,6
2,C00001,"Ohta Co.,Ltd.",798648,DF410848-02,2026-01-22,360,0,2026-06-27,360,New,2026,6
3,C00001,"Ohta Co.,Ltd.",798650,DF410847,2026-01-22,180,0,2026-06-27,180,New,2026,6
4,C00001,"Ohta Co.,Ltd.",798652,DF309392,2026-01-22,180,0,2026-06-27,180,New,2026,6


### 4. SALE_PRICE

In [14]:
df_sale_price.head() # Display the first few rows of the sale price DataFrame
df_sale_price.info() # Get information about the sale price DataFrame, including data types and non-null counts

,Document No,Customer Code,Currency,Item Code,Unit,Price Type,Quantity,Unit Price,Unit Price VND,Date
0,Item Master,C01138,VND,PX21.8,Pcs,Retail,1,"68,425.0000","68,425.0000",2026-03-25
1,Item Master,C01138,VND,SS6KW,Pcs,Follow Q'ty,5,"309,150.0000","309,150.0000",2026-03-25
2,Item Master,C01138,VND,SS12K,Pcs,Retail,1,"268,650.0000","268,650.0000",2026-03-25
3,Item Master,C01138,VND,SC4Y,Pcs,Retail,1,"268,650.0000","268,650.0000",2026-03-25
4,Item Master,C00173,USD,X56960021,Pcs,SPQ,30000,0.2531,0.2531,2026-03-25


<class 'pandas.DataFrame'>
RangeIndex: 85092 entries, 0 to 85091
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Document No     85092 non-null  object        
 1   Customer Code   85092 non-null  str           
 2   Currency        85092 non-null  str           
 3   Item Code       85092 non-null  object        
 4   Unit            85092 non-null  str           
 5   Price Type      85092 non-null  str           
 6   Quantity        85092 non-null  int64         
 7   Unit Price      85092 non-null  float64       
 8   Unit Price VND  85092 non-null  float64       
 9   Date            85092 non-null  datetime64[us]
dtypes: datetime64[us](1), float64(2), int64(1), object(2), str(4)
memory usage: 6.5+ MB


In [15]:
# Filter item code with the latest sale price for each item
df_sale_price_filtered = df_sale_price.sort_values('Date').drop_duplicates(subset=['Item Code','Customer Code'], keep='last')
df_sale_price_filtered.head() # Display the first few rows of the filtered sale price DataFrame

,Document No,Customer Code,Currency,Item Code,Unit,Price Type,Quantity,Unit Price,Unit Price VND,Date
85004,QUO-C00476-EMP068-019,C00476,VND,5100150-16+35-029,Pcs,Retail,8,"13,550.0000","13,550.0000",2024-01-02
85005,QUO-C00476-EMP068-019,C00476,VND,7100200-16-001,Pcs,Retail,8,"2,839.0000","2,839.0000",2024-01-02
85006,QUO-C00476-EMP068-019,C00476,VND,7200200-16-064,Pcs,Retail,8,"9,387.0000","9,387.0000",2024-01-02
85007,QUO-C00476-EMP068-019,C00476,VND,5100150-18+50-047,Pcs,Retail,6,"62,084.0000","62,084.0000",2024-01-02
85008,QUO-C00476-EMP068-019,C00476,VND,7100200-18-002,Pcs,Retail,6,"22,814.0000","22,814.0000",2024-01-02


### 5. Track_PO

In [16]:
df_track_po.info()

<class 'pandas.DataFrame'>
RangeIndex: 32741 entries, 0 to 32740
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Supplier Code        32741 non-null  str           
 1   Supplier Name        32741 non-null  str           
 2   PO No                32741 non-null  str           
 3   Shipping Method      32741 non-null  str           
 4   PO Date              32741 non-null  datetime64[us]
 5   Item Code            32741 non-null  object        
 6   PO Qty               32737 non-null  float64       
 7   Received Qty         32737 non-null  float64       
 8   Remain Qty           32737 non-null  float64       
 9   Receiving Plan Date  32587 non-null  object        
 10  PO Status            32737 non-null  str           
 11  Amount               32737 non-null  float64       
 12  Customer Code        32737 non-null  str           
dtypes: datetime64[us](1), float64(4), object(2

In [17]:
df_track_po["Receiving Plan Date"] = pd.to_datetime(df_track_po["Receiving Plan Date"])

df_track_po.head()

,Supplier Code,Supplier Name,PO No,Shipping Method,PO Date,Item Code,PO Qty,Received Qty,Remain Qty,Receiving Plan Date,PO Status,Amount,Customer Code
0,S00825,"Nhat Dang Quang CO.,LTD.",C00895-EMP152-230103-01,Car,2023-01-03,CABLE-TIE-300+5,50.0000,50.0000,0.0000,2023-01-06,Closed,"1,250,000.0000",C00895
1,S00825,"Nhat Dang Quang CO.,LTD.",C00895-EMP152-230103-01,Car,2023-01-03,CABLE-TIE-500+8,1.0000,1.0000,0.0000,2023-01-06,Closed,"59,000.0000",C00895
2,S00721,"Kim Thiet Manufacturing Co., Ltd",C00664-EMP063-230103-01,Car,2023-01-03,014-135-02-C-02,"3,059.0000","3,059.0000",0.0000,2023-01-09,Closed,"11,624,200.0000",C00664
3,S00721,"Kim Thiet Manufacturing Co., Ltd",C00664-EMP063-230103-01,Car,2023-01-03,014-135-02-C-GRAY-02,3.0000,3.0000,0.0000,2023-01-06,Closed,"11,400.0000",C00664
4,S00721,"Kim Thiet Manufacturing Co., Ltd",C00664-EMP063-230103-01,Car,2023-01-03,014-135-02-C-GRAY-02,"2,280.0000","2,280.0000",0.0000,2023-01-09,Closed,"8,664,000.0000",C00664


# III. DATA ANALYSIS

### Module 1: Inventory Status and ABC Analysis

#### 1. Assign the stock status of each item code

In [18]:
# Group the inventory data by "Item Code" and calculate the total available stock for each item
df_inventory_sum = df_inventory.groupby(["Item Code"]).agg(
    {"Item Group"      : "first",
     "Item Category"   : "first",
     "Sub Category"    : "first",  
     "Unit"            : "first",  
     "Minimum Stock"   : "max",
     "Reservation"     : "max",
     "Available"       : "sum",
     "OK"              : "sum",
     "QA Waiting"      : "sum",
     "Remade"          : "sum",
     "NG"              : "sum",
     "Total"           : "sum"
    }  
).reset_index()


Classify the stock status as below table:


| Conditions | Stock Status |
|------------|--------------|
|Available = 0 | Stockout|
|0 < Available < Min Stock| Stockout Risk|
| Available >= Min Stock and Min Stock > 0 | Normal |
| Min Stock = 0 | No min set
| Available > 3 * Min Stock | Excess|
 

In [19]:
conditions = [  df_inventory_sum["Minimum Stock"] == 0,             
               (df_inventory_sum["Available"] > 0) & (df_inventory_sum["Available"] < df_inventory_sum["Minimum Stock"]),
               (df_inventory_sum["Available"] >= df_inventory_sum["Minimum Stock"]) & (df_inventory_sum["Minimum Stock"] > 0),
               (df_inventory_sum["Available"] > df_inventory_sum["Minimum Stock"]*3),
                 df_inventory_sum["Available"] == 0
               ] 

stock_status = ["no min set", "stockout risk", "normal", "excess", "stockout"]

df_inventory_sum["Stock Status"] = np.select(conditions, stock_status, default = "unknown")

df_inventory_sum.head()

    

,Item Code,Item Group,Item Category,Sub Category,Unit,Minimum Stock,Reservation,Available,OK,QA Waiting,Remade,NG,Total,Stock Status
0,304,Internal Finish Products,Internal Finish Products,Internal Finish Products,Pcs,0.0000,0,3.0000,3.0000,0,0.0000,0,3.0000,no min set
1,1120,Screws,Tools,Tools,Pairs,0.0000,0,8.0000,8.0000,0,0.0000,0,8.0000,no min set
2,13005,Screws,Tools,Tools,Pcs,0.0000,0,9.0000,9.0000,0,0.0000,0,9.0000,no min set
3,13013,Screws,Tools,Tools,Pcs,0.0000,0,3.0000,3.0000,0,0.0000,0,3.0000,no min set
4,13021,Screws,Tools,Tools,Pcs,0.0000,0,1.0000,1.0000,0,0.0000,0,1.0000,no min set


In [20]:
df_inventory_sum['NG Rate'] = np.where(df_inventory_sum['Total'] == 0, 0, (df_inventory_sum['NG'] + df_inventory_sum['Remade']) / df_inventory_sum['Total'] * 100)

df_inventory_sum.head()

,Item Code,Item Group,Item Category,Sub Category,Unit,Minimum Stock,Reservation,Available,OK,QA Waiting,Remade,NG,Total,Stock Status,NG Rate
0,304,Internal Finish Products,Internal Finish Products,Internal Finish Products,Pcs,0.0000,0,3.0000,3.0000,0,0.0000,0,3.0000,no min set,0.0000
1,1120,Screws,Tools,Tools,Pairs,0.0000,0,8.0000,8.0000,0,0.0000,0,8.0000,no min set,0.0000
2,13005,Screws,Tools,Tools,Pcs,0.0000,0,9.0000,9.0000,0,0.0000,0,9.0000,no min set,0.0000
3,13013,Screws,Tools,Tools,Pcs,0.0000,0,3.0000,3.0000,0,0.0000,0,3.0000,no min set,0.0000
4,13021,Screws,Tools,Tools,Pcs,0.0000,0,1.0000,1.0000,0,0.0000,0,1.0000,no min set,0.0000


#### 2. ABC Analysis

In [21]:
# Merge SO and sale price data to get the unit price for each sales order line
df_so_sale = pd.merge(df_so, df_sale_price_filtered[["Item Code", "Customer Code", "Unit Price VND"]], on=["Item Code", "Customer Code"], how="left")

df_so_sale.head()

,Customer Code,Customer Name,SO No,Item Code,Order Date,Order Qty,Delivery Qty,Delivery Date,Remain Quantity,SO Status,Delivery Year,Delivery Month,Unit Price VND
0,C00001,"Ohta Co.,Ltd.",801104,M1TM1851-A01-02,2026-02-16,400,0,2026-08-26,400,New,2026,8,611.0000
1,C00001,"Ohta Co.,Ltd.",798646,DF410842-A,2026-01-22,180,0,2026-06-27,180,New,2026,6,"4,091.0000"
2,C00001,"Ohta Co.,Ltd.",798648,DF410848-02,2026-01-22,360,0,2026-06-27,360,New,2026,6,996.0000
3,C00001,"Ohta Co.,Ltd.",798650,DF410847,2026-01-22,180,0,2026-06-27,180,New,2026,6,"1,473.0000"
4,C00001,"Ohta Co.,Ltd.",798652,DF309392,2026-01-22,180,0,2026-06-27,180,New,2026,6,"5,828.0000"


In [22]:
# Calculate the total sales amount for each sales order line
df_so_sale["Revenue"]                = df_so_sale["Order Qty"] * df_so_sale["Unit Price VND"]

# Choose the so issued in 2024 and 2025 for ABC Analysis
df_so_sale_filtered                  = df_so_sale[df_so_sale["Delivery Year"].isin([2024, 2025])]

# Sum the total amount for each item over the years
df_so_sale_sum                       = df_so_sale_filtered.groupby(["Delivery Year", "Item Code"]).agg(
                                       Total_Quantity = ("Order Qty", "sum"),
                                       Total_Revenue  = ("Revenue"  , "sum") 
                                       ).reset_index()

# Calculate the cumulative revenue and the percentage of total revenue for each item
df_abc                              = df_so_sale_sum.sort_values("Total_Revenue", ascending=False).reset_index(drop=True)
df_abc["Cumulative_Revenue"]        = df_abc["Total_Revenue"].cumsum()
total_all_revenue = df_abc['Total_Revenue'].sum()
df_abc['Cumulative_Percentage']     = (df_abc['Cumulative_Revenue'] / total_all_revenue) * 100

df_abc.head()

,Delivery Year,Item Code,Total_Quantity,Total_Revenue,Cumulative_Revenue,Cumulative_Percentage
0,2025,TRP-20429-03-02,2000,"5,692,700,000.0000","5,692,700,000.0000",6.1497
1,2024,TRP-20429-03-02,847,"2,410,858,450.0000","8,103,558,450.0000",8.7541
2,2024,C99-778140-02,790000,"2,365,260,000.0000","10,468,818,450.0000",11.3092
3,2025,C99-778140-02,700000,"2,095,800,000.0000","12,564,618,450.0000",13.5733
4,2024,FS-M4-0,1300000,"1,848,600,000.0000","14,413,218,450.0000",15.5703


In [23]:
# classify items into A, B, C categories based on cumulative percentage
conditions_abc                = [
                                    df_abc["Cumulative_Percentage"] <= 80,
                                    (df_abc["Cumulative_Percentage"] > 80) & (df_abc["Cumulative_Percentage"] <= 95),
                                    df_abc["Cumulative_Percentage"] > 95
                                ]

abc_category                  = ["A", "B", "C"]
df_abc["ABC_Category"]        = np.select(conditions_abc, abc_category, default="Unknown")

# Check the total count and total revenue for each ABC category
abc_summary                   = df_abc.groupby("ABC_Category").agg(
                                                Total_Items = ("Item Code", "count"),
                                                Total_Revenue = ("Total_Revenue", "sum")
                                              ).reset_index()

abc_summary['revenue_share'] = (abc_summary['Total_Revenue'] / total_all_revenue) * 100

abc_summary['items_share']   = (abc_summary['Total_Items'] / abc_summary['Total_Items'].sum()) * 100

df_abc
abc_summary

# If the revenue_share_% of A: 80%, B: 15%, C: 5%, then the ABC classification is reasonable. If not, the thresholds for ABC classification can be adjusted to achieve a more balanced distribution of items across the categories.


,Delivery Year,Item Code,Total_Quantity,Total_Revenue,Cumulative_Revenue,Cumulative_Percentage,ABC_Category
0,2025,TRP-20429-03-02,2000,"5,692,700,000.0000","5,692,700,000.0000",6.1497,A
1,2024,TRP-20429-03-02,847,"2,410,858,450.0000","8,103,558,450.0000",8.7541,A
2,2024,C99-778140-02,790000,"2,365,260,000.0000","10,468,818,450.0000",11.3092,A
3,2025,C99-778140-02,700000,"2,095,800,000.0000","12,564,618,450.0000",13.5733,A
4,2024,FS-M4-0,1300000,"1,848,600,000.0000","14,413,218,450.0000",15.5703,A
...,...,...,...,...,...,...,...
6962,2024,2520200-5+12-057,30,0.0000,"92,568,897,240.0548",100.0000,C
6963,2024,1200111-3+15-002,3000,0.0000,"92,568,897,240.0548",100.0000,C
6964,2025,C011463-01-02,480,0.0000,"92,568,897,240.0548",100.0000,C
6965,2024,2520200-4+12-028,30,0.0000,"92,568,897,240.0548",100.0000,C


,ABC_Category,Total_Items,Total_Revenue,revenue_share,items_share
0,A,576,"74,031,369,740.9900",79.9743,8.2675
1,B,1202,"13,905,468,524.3034",15.0218,17.2528
2,C,5189,"4,632,058,974.7615",5.0039,74.4797


#### 3. Check the stock status of each item after classification

This step filters items A and B contributing the high revenue to the company and considers to remove C which slow-moving

In [24]:
# Convert the ABC classification results into a pivot table format for better visualization and analysis

df_abc_pivot = df_abc[["Item Code", "Delivery Year", "Cumulative_Percentage","ABC_Category"]].pivot_table(
                        index    = "Item Code", 
                        columns  = "Delivery Year", 
                        values   = ["Cumulative_Percentage", "ABC_Category"], 
                        aggfunc  = "first"
                        ).reset_index()


df_abc_pivot.head()

Item Code ABC_Category      Cumulative_Percentage        
Delivery Year                     2024 2025                  2024    2025
0                     179            B  NaN               86.7890     NaN
1                    1120            C  NaN               99.5915     NaN
2                   13013          NaN    C                   NaN 99.9436
3                   14001          NaN    C                   NaN 99.4731
4             22,050.0208            C  NaN               99.7801     NaN

In [25]:

df_abc_pivot.columns      = ["Item Code", "ABC_2024","ABC_2025","Cumulative_Pct_2024", "Cumulative_Pct_2025"]
df_abc_pivot              = df_abc_pivot[["Item Code", "ABC_2024", "Cumulative_Pct_2024", "ABC_2025", "Cumulative_Pct_2025"]]

df_abc_pivot_inv          = pd.merge(df_abc_pivot, df_inventory_sum, on = "Item Code", how="outer")

columns_filtered          = ["Item Code","Item Group", "Item Category",
                             "Minimum Stock", "Reservation", "Available", "OK", "QA Waiting", "Remade", "NG", "Total", "NG Rate", "Stock Status",
                            "ABC_2024", "Cumulative_Pct_2024", "ABC_2025", "Cumulative_Pct_2025"]              
                             
                             

df_abc_pivot_inv_filtered = df_abc_pivot_inv[columns_filtered]
df_abc_pivot_inv_filtered.head()

,Item Code,Item Group,Item Category,Minimum Stock,Reservation,Available,OK,QA Waiting,Remade,NG,Total,NG Rate,Stock Status,ABC_2024,Cumulative_Pct_2024,ABC_2025,Cumulative_Pct_2025
0,179,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,B,86.7890,NaN,NaN
1,304,Internal Finish Products,Internal Finish Products,0.0000,0.0000,3.0000,3.0000,0.0000,0.0000,0.0000,3.0000,0.0000,no min set,NaN,NaN,NaN,NaN
2,1120,Screws,Tools,0.0000,0.0000,8.0000,8.0000,0.0000,0.0000,0.0000,8.0000,0.0000,no min set,C,99.5915,NaN,NaN
3,13005,Screws,Tools,0.0000,0.0000,9.0000,9.0000,0.0000,0.0000,0.0000,9.0000,0.0000,no min set,NaN,NaN,NaN,NaN
4,13013,Screws,Tools,0.0000,0.0000,3.0000,3.0000,0.0000,0.0000,0.0000,3.0000,0.0000,no min set,NaN,NaN,C,99.9436


#### 4. Action

##### Action 1: Send to the procurement department the list items A with stockout or stockout risk for urgent buying

In [26]:
# Filtered the items A in 2025 with stockout risk or stockout status to prioritize the inventory management and procurement planning for these items, ensuring that they are available to meet customer demand and avoid potential lost sales or customer dissatisfaction. 
df_action_A = df_abc_pivot_inv[
    (df_abc_pivot_inv["ABC_2025"] == "A") &
    (df_abc_pivot_inv["Stock Status"].isin(["stockout", "stockout risk"]))
    ]

df_action_A_customer = pd.merge(df_action_A, df_sale_price_filtered[["Item Code", "Customer Code", "Unit Price VND"]], on="Item Code", how="left")

# List the items that need to be prioritized for procurement and inventory management based on their ABC classification and stock status, along with the relevant information such as available stock, reservation, minimum stock, NG rate, and unit price for each customer.
list_buy_A = df_action_A_customer.groupby(["Item Code", "Item Group", "Customer Code"]).agg(
    Available        = ("Available", "sum"),
    Reservation      = ("Reservation", "max"),
    Minimum_Stock    = ("Minimum Stock", "first"),
    NG_Rate          = ("NG Rate", "mean"),
    ABC_2025         = ("ABC_2025", "first"),
    Stock_Status     = ("Stock Status", "first")
).reset_index()

list_buy_A.head()

,Item Code,Item Group,Customer Code,Available,Reservation,Minimum_Stock,NG_Rate,ABC_2025,Stock_Status
0,503290,Screws,C00056,3.0000,0.0000,25.0000,0.0000,A,stockout risk
1,1400111-3+08-002,Screws,C00030,"53,500.0000",0.0000,"64,800.0000",0.0000,A,stockout risk
2,1400111-5+08-002,Screws,C00808,"1,706.0000",0.0000,"11,200.0000",0.0000,A,stockout risk
3,1400111-5+08-002,Screws,C00942,"1,706.0000",0.0000,"11,200.0000",0.0000,A,stockout risk
4,1400111-5+08-002,Screws,C01460,"1,706.0000",0.0000,"11,200.0000",0.0000,A,stockout risk


##### Action 2: Reporting to the Sales Department on the items C with excess, less demand 

In [27]:
# 1. Filtered the items C in in 2025 with excess stock status to identify the items that are 
# overstocked and belong to the C category, which typically represents low-value or low-priority items. 
# This information can be used to develop strategies for managing excess inventory, such as offering discounts, 
# bundling with other products, or discontinuing the items to free up storage space and reduce carrying costs.
df_action_C = df_abc_pivot_inv_filtered[
    (df_abc_pivot_inv_filtered['ABC_2025']     == 'C')]



df_action_C_customer = pd.merge(df_action_C, df_sale_price_filtered[["Item Code", "Customer Code", "Unit Price VND"]], on="Item Code", how="left")

# 2. Group the items by Item Code and Customer Code to send to Sales for handling and salvaging the capital tied up in excess inventory.
list_to_clear_C      = df_action_C_customer.groupby(['Item Code', 'Customer Code']).agg(
    Available        = ("Available", "sum"),
    Reservation      = ("Reservation", "max"),
    Minimum_Stock    = ("Minimum Stock", "first"),
    NG_Rate          = ("NG Rate", "mean"),
    ABC_2025         = ("ABC_2025", "first"),
    Stock_Status     = ("Stock Status", "first")
).reset_index()

list_to_clear_C.head()

# list_to_clear_C.to_excel(os.path.join(path, "list_to_clear_C.xlsx"), index=False)


,Item Code,Customer Code,Available,Reservation,Minimum_Stock,NG_Rate,ABC_2025,Stock_Status
0,13013,C00030,3.0000,0.0000,0.0000,0.0000,C,no min set
1,13013,C01015,3.0000,0.0000,0.0000,0.0000,C,no min set
2,13013,C01115,3.0000,0.0000,0.0000,0.0000,C,no min set
3,13013,C01478,3.0000,0.0000,0.0000,0.0000,C,no min set
4,14001,C00030,7.0000,0.0000,0.0000,0.0000,C,no min set


### Module 2: Stock vs Demand Gap

In [28]:
# Filtered SO new and in process
condition_status                = ["New", "In Process"]


list_so_not_completed           = df_so_sale_filtered[df_so_sale_filtered['SO Status'].isin(condition_status)]

so_not_completed_sum            = list_so_not_completed.groupby(["Customer Code", "Item Code"]).agg(
                                                                Total_Remain  = ("Remain Quantity", "sum"),

                                                                 ).reset_index()

so_inventory                    = pd.merge(so_not_completed_sum, df_inventory_sum, on="Item Code", how="left")

so_inventory[main_quantity]     = so_inventory[main_quantity].fillna(0)
so_inventory["Minimum Stock"]   = so_inventory["Minimum Stock"].fillna("no min set")
so_inventory["Stock Status"]    = so_inventory["Stock Status"].fillna("stockout")

so_inventory["Gap"]             = so_inventory["Reservation"] + so_inventory["Available"] - so_inventory["Total_Remain"]

# Filtered the sales orders that are not completed and have a gap between the available stock and the remaining quantity in the sales order, 
# which can help prioritize inventory management and procurement planning to ensure that there is enough stock to 
# fulfill customer orders and avoid potential lost sales or customer dissatisfaction.


so_inventory_lack               = so_inventory[so_inventory["Gap"] < 0].sort_values(by = "Gap", ascending=True)

so_inventory_lack.head(5)
# so_inventory_lack.to_excel(os.path.join(path, "so_lack.xlsx"), index=False)

,Customer Code,Item Code,Total_Remain,Item Group,Item Category,Sub Category,Unit,Minimum Stock,Reservation,Available,OK,QA Waiting,Remade,NG,Total,Stock Status,NG Rate,Gap
187,C00056,DKP048-01011-01-0-02,23000,OutSource Processing,OutSource Processing,OutSource Processing,Pcs,"4,500.0000","2,200.0000","5,709.0000","7,909.0000",0.0000,0.0000,0.0000,"7,909.0000",normal,0.0000,"-15,091.0000"
44,C00030,1400111-3+08-002,64800,Screws,Machine Screw,(+) Binding Screw,Pcs,"64,800.0000",0.0000,"53,500.0000","53,500.0000",0.0000,0.0000,0.0000,"53,500.0000",stockout risk,0.0000,"-11,300.0000"
0,C00002,2230130-6+16-002,5912,Screws,Sems Screw,(+) Upset P=3,Pcs,"5,100.0000",0.0000,838.0000,838.0000,0.0000,0.0000,30.0000,868.0000,stockout risk,3.4562,"-5,074.0000"
108,C00030,3150111-3+08-002,60000,Screws,Tap Tite/High Technology Screw,(+) P-Tite Pan,Pcs,"65,000.0000",0.0000,"55,000.0000","55,000.0000",0.0000,0.0000,0.0000,"55,000.0000",stockout risk,0.0000,"-5,000.0000"
68,C00030,2130111-3+06-002,45000,Screws,Sems Screw,(+) Pan P=3,Pcs,"45,075.0000",0.0000,"41,350.0000","41,350.0000",0.0000,0.0000,0.0000,"41,350.0000",stockout risk,0.0000,"-3,650.0000"


### Module 3: SO BACKLOG
This module shows the againg days of SO

| Days_Overdue | Delay Status |
|------------|--------------|
|>= 7 | slightly late|
|7-15| moderately late|
| >15 | severely late |


#### 1. Delay classification level

In [29]:
list_so_not_completed["Days_Overdue"] = (list_so_not_completed["Delivery Date"] - pd.Timestamp("2026-03-10")).dt.days

delay_status                          = ["slightly late", "moderately late", "severely late"]

so_late                               = list_so_not_completed[list_so_not_completed["Days_Overdue"] < 0]

so_late                               = so_late[so_late["Delivery Date"] >= pd.Timestamp("2025-06-01")]

conditions_delay_status               = [
                                            (so_late["Days_Overdue"] >= -7),
                                            (so_late["Days_Overdue"] < -7) & (so_late["Days_Overdue"] >= -15),
                                            (so_late["Days_Overdue"] < -15)
                                        ]


so_late["Delay_Status"]               = np.select(conditions_delay_status, delay_status, default="on time")

df_backlog                            = pd.merge(
                                                    so_late, df_abc_pivot_inv_filtered[["Item Code", "Stock Status", "ABC_2025"]], 
                                                    on = "Item Code", how = "left"
                                                )

# Items with no classfication will be "N", which is as considered as "B"
df_backlog["ABC_2025"]                = df_backlog["ABC_2025"].fillna("N")

# Calculate the backlog value
df_backlog["Backlog_Value"]           = df_backlog["Remain Quantity"] * df_backlog["Unit Price VND"]
df_backlog.head(2)


,Customer Code,Customer Name,SO No,Item Code,Order Date,Order Qty,Delivery Qty,Delivery Date,Remain Quantity,SO Status,Delivery Year,Delivery Month,Unit Price VND,Revenue,Days_Overdue,Delay_Status,Stock Status,ABC_2025,Backlog_Value
0,C00030,"TAKAZONO VIETNAM Co.,Ltd.",FC-VMI-X3,1100111-3+04-002,2025-08-01,9000,0,2025-12-31,9000,New,2025,12,135.0000,"1,215,000.0000",-69,severely late,normal,C,"1,215,000.0000"
1,C00030,"TAKAZONO VIETNAM Co.,Ltd.",FC-VMI-X3,1100111-3+06-002,2025-08-01,7500,0,2025-12-31,7500,New,2025,12,155.0000,"1,162,500.0000",-69,severely late,stockout risk,B,"1,162,500.0000"


#### 2. Action

Identify item codes that need to be prioritized for production to fulfill urgent customer deliveries.

In [30]:
backlog_summary     = df_backlog.groupby(
                                        ['Customer Code', 'SO No','Item Code', 'ABC_2025', 'Delay_Status', "Delivery Date", "Days_Overdue"]).agg(
                                        Total_Backlog_Qty=('Remain Quantity', 'sum'),      
                                        Total_Backlog_Value=('Backlog_Value', 'sum')                                              
                                        ).reset_index()

backlog_summary     = backlog_summary.sort_values(by='Total_Backlog_Value', ascending=False)

backlog_summary.head(5)

,Customer Code,SO No,Item Code,ABC_2025,Delay_Status,Delivery Date,Days_Overdue,Total_Backlog_Qty,Total_Backlog_Value
160,C00056,Draft - so for koji san email,K1-K131-20-001,A,severely late,2025-11-19,-111,46000,"369,932,000.0000"
157,C00056,Draft - so for koji san email,DKP048-01011-01-0-02,A,severely late,2025-11-19,-111,23000,"186,300,000.0000"
154,C00056,Draft - so for koji san email,6240200-6-002,A,severely late,2025-11-19,-111,23000,"88,274,000.0000"
158,C00056,Draft - so for koji san email,DLK1601122-02,A,severely late,2025-11-19,-111,1800,"56,520,000.0000"
159,C00056,Draft - so for koji san email,DLK1601143-02,A,severely late,2025-11-19,-111,2700,"41,580,000.0000"


Action 1: Item Codes with A in severely late or moderately late must be immediately produced to fullfill customer order. This group bring the highest revenue to the company, so Purchasing and Logistic departments need to arrange the plan.

Action 2: Item Codes with B in severely late or moderately late will be manufactured after A finished or Production Dept should scheduled overlap with B is possible. The Sales Dept actively contacts to the customers for enlarging the delivery date.

## Module 4. Lead Time, Set up Reorder Point, Min/Max levels

In [31]:
# Prepare the data

po_closed_item                     = pd.merge(df_track_po[df_track_po["PO Status"] == "Closed"], 
                                              df_item_filtered[["Item Code", "Item Type", "Item Group"]],
                                              on = "Item Code", how = "left")

po_closed_item["Lead_Time"]        = (po_closed_item["Receiving Plan Date"] - po_closed_item["PO Date"]).dt.days


# Filtered closed SO in 2025 from df_so_sale

so_closed_2025                      = df_so_sale_filtered[
                                      (df_so_sale_filtered["Delivery Year"] == 2025) &
                                      (df_so_sale_filtered["SO Status"]    == "Closed")
                                                       ]

so_closed_2025_item                 = pd.merge(so_closed_2025, df_item_filtered[["Item Code"]], how  = "left", on = "Item Code")
so_closed_2025_item["Delivery Qty"] = so_closed_2025_item["Delivery Qty"].fillna(0) 

# Assign ABC Class in so_closed_2025
# so_closed_2025_abc              = pd.merge(so_closed_2025, df_abc_pivot_inv_filtered[["Item Code", "ABC_2025"]], how = "left", on = "Item Code")

so_closed_2025_sum                  = so_closed_2025_item.groupby(["Item Code", "Delivery Month"]).agg(
                                                          Total_Qty_Month_2025 = ("Delivery Qty", "sum")
                                                          ).reset_index()

so_closed_2025_sum.head(5)

,Item Code,Delivery Month,Total_Qty_Month_2025
0,13013,10,1
1,14001,1,10
2,73115,10,1
3,73795,10,1
4,407209,5,400


##### 1. Identify demand rate (Regular / Seasonal / Sporadic) to classify XYZ for each items
Regular  : ≥ 80%
Seasonal : 40 - 80%
Sporadic : < 40%

Assign XYZ according to the below rules:
X : CV <= 0.5
Y: CV from 0.5-1.0
Z: CV > 1.0

In [32]:
# Demand Rate of each item in 12 months of 2025
demand_freq                     = so_closed_2025_sum.groupby("Item Code").agg(
                                                        Count_Month_2025 = ("Delivery Month"       , "count"),
                                                        Total_Qty_SO     = ("Total_Qty_Month_2025" , "sum")
                                                            ).reset_index()
                     
demand_freq["Demand_Rate"]      = (demand_freq["Count_Month_2025"]/12).round(4)



# Identify demand rate
demand_category                = ["Regular", "Seasonal", "Sporadic"]

demand_conditions              = [demand_freq["Demand_Rate"] >= 0.8,
                                  (demand_freq["Demand_Rate"] >= 0.4) & (demand_freq["Demand_Rate"] < 0.8),
                                  (demand_freq["Demand_Rate"] < 0.4)
                                 ]

demand_freq["Demand_Class"]    = np.select(demand_conditions, demand_category, default = "not determined" )   




In [33]:
# Calculate CV of regular items
regular_items                          = demand_freq[demand_freq["Demand_Class"] == "Regular"]["Item Code"]
df_regular                             = so_closed_2025_sum[so_closed_2025_sum["Item Code"].isin(regular_items)]

df_regular_pivot                       = df_regular.pivot_table(index   = "Item Code", 
                                                        columns = "Delivery Month", 
                                                        values  = "Total_Qty_Month_2025",
                                                        aggfunc = "sum"
                                            ).reindex(columns   = range(1, 13), fill_value = 0).fillna(0)

df_regular_pivot.index.name            = "Item Code" 

df_regular_pivot["Avg_Monthly"]        = df_regular_pivot.mean(axis = 1)
df_regular_pivot["Std_Monthly"]        = df_regular_pivot.std(axis = 1)
df_regular_pivot["CV"]                 = df_regular_pivot["Std_Monthly"] / df_regular_pivot["Avg_Monthly"]
df_regular_sum                         = df_regular_pivot[["Std_Monthly", "Avg_Monthly", "CV"]].reset_index()
df_regular_sum.columns.name = None

#df_regular_pivot.head()

# Calcilate CV of seasonal items
seasonal_items                         = demand_freq[demand_freq["Demand_Class"] == "Seasonal"]["Item Code"]
df_seasonal                            = so_closed_2025_sum[so_closed_2025_sum["Item Code"].isin(seasonal_items)]

df_seasonal_sum                        = df_seasonal.groupby("Item Code")["Total_Qty_Month_2025"].agg(Avg_Monthly = "mean", Std_Monthly = "std").reset_index()
df_seasonal_sum["CV"]                  = df_seasonal_sum["Std_Monthly"] / df_seasonal_sum["Avg_Monthly"]

# Class X, Y, Z
xyz_class                              = ["X", "Y", "Z"]

def assign_xyz(df):
    df["XYZ"] = pd.cut(
        df["CV"],
        bins   = [-float('inf'), 0.5,1.0, float('inf')],
        labels = ['X', 'Y', 'Z']
    )
    return df
         


df_seasonal_sum                        = assign_xyz(df_seasonal_sum)
df_regular_sum                         = assign_xyz(df_regular_sum)

# Merge 2 dataframes
sea_reg                                = pd.concat([df_seasonal_sum, df_regular_sum], ignore_index = True) 

demand_freq_all                        = pd.merge(demand_freq, sea_reg, how = "left", on = "Item Code")
demand_freq_all["XYZ"]                 = demand_freq_all["XYZ"].fillna("Z")

demand_freq_all["Avg_Daily"]           = demand_freq_all["Avg_Monthly"]/30
demand_freq_all["Std_Daily"]           = demand_freq_all["Std_Monthly"]/30


#### 2. Identify ABC x XYZ

In [34]:
df_abc_xyz                             = pd.merge(demand_freq_all, df_abc_pivot_inv_filtered[["Item Code", "ABC_2025"]], on = "Item Code", how = "left")
len(df_abc_xyz)

3577

#### 3. Lead Time Per Item

In [35]:
df_track_po["PO_Month"]           = df_track_po["PO Date"].dt.month
df_track_po["PO_Year"]            = df_track_po["PO Date"].dt.year.fillna(0).astype(int)


df_track_po["Receiving_Year"]     = df_track_po["Receiving Plan Date"].dt.year.fillna(0).astype(int)
df_track_po["Receiving_Month"]    = df_track_po["Receiving Plan Date"].dt.month.fillna(0).astype(int)

df_track_po["Lead_Time"]          = (df_track_po["Receiving Plan Date"] - df_track_po["PO Date"]).dt.days.astype("Int64")

# Filter PO 

po_closed_2025                    = df_track_po[ (df_track_po["PO_Year"]    == 2025) &
                                                 (df_track_po["PO Status"] == "Closed")
                                               ]

# Count PO per item + Supplier to show PO with the most POs per item. That is the primary supplier.
primary_supplier                  = po_closed_2025.groupby(["Item Code", "Supplier Code"]).agg(
                                                      Qty_PO = ("PO No"     ,"count"),
                                                      P90_LT = ("Lead_Time" ,lambda x: x.quantile(0.9))
                                                ).reset_index()\
                                                .sort_values("Qty_PO", ascending = False)\
                                                .drop_duplicates(subset = "Item Code", keep = "first")\
                                                .reset_index(drop=True)
# Lead time supplier for PO < 10
lt_supplier                       = po_closed_2025.groupby("Supplier Code")["Lead_Time"].quantile(0.9).reset_index()
lt_supplier.columns               = ['Supplier Code', 'LT_P90_Supplier']



primary_supplier                  = pd.merge(primary_supplier, lt_supplier, on = "Supplier Code", how="left")

# Choose final L90
primary_supplier['LT_P90_Final']  = primary_supplier.apply(lambda row: row['P90_LT'] if row['Qty_PO'] >= 10 else row['LT_P90_Supplier'], axis=1)
                                        

item_class                        = pd.merge(primary_supplier, df_abc_xyz, how = "left", on = "Item Code")
item_class["General_Class"]       = item_class["ABC_2025"].astype(str) + item_class["XYZ"].astype(str)  

item_class.head()

,Item Code,Supplier Code,Qty_PO,P90_LT,LT_P90_Supplier,LT_P90_Final,Count_Month_2025,Total_Qty_SO,Demand_Rate,Demand_Class,Avg_Monthly,Std_Monthly,CV,XYZ,Avg_Daily,Std_Daily,ABC_2025,General_Class
0,AAFK73C0-02,S00197,39,9.4000,7.0000,9.4000,10.0000,"1,965.0000",0.8333,Regular,163.7500,166.5113,1.0169,Z,5.4583,5.5504,C,CZ
1,MS-TP-8+8-M4-019,S00034,31,11.0000,13.0000,11.0000,11.0000,"51,542.0000",0.9167,Regular,"4,295.1667","3,402.8447",0.7922,Y,143.1722,113.4282,A,AY
2,ALN-4-005,S00034,30,18.1000,13.0000,18.1000,12.0000,"830,000.0000",1.0000,Regular,"69,166.6667","15,523.2800",0.2244,X,"2,305.5556",517.4427,C,CX
3,K1-K131-20-001,S00001,30,30.4000,20.8000,30.4000,12.0000,"55,810.0000",1.0000,Regular,"4,650.8333","2,187.7898",0.4704,X,155.0278,72.9263,A,AX
4,8480150-3-005,S00034,30,18.1000,13.0000,18.1000,12.0000,"1,388,000.0000",1.0000,Regular,"115,666.6667","22,528.9937",0.1948,X,"3,855.5556",750.9665,C,CX


#### 4. Safety Stock, Reorder Point, Min/Max

In [36]:
z_map                                       = {
                                                'AX': 1.65, 'AY': 1.65, 'AZ': 1.65,
                                                'BX': 1.28, 'BY': 1.28, 'BZ': 1.28,
                                                'CX': None, 'CY': None, 'CZ': None
                                                }

item_class['Z_Score']                       = item_class['General_Class'].map(z_map)
item_class["Safety_Stock"]                  = item_class["Z_Score"] * item_class["Std_Daily"] * (item_class["LT_P90_Final"] ** 0.5)
item_class["Reorder_Point"]                 = item_class["Avg_Daily"] * item_class["LT_P90_Final"] + item_class["Safety_Stock"]
item_class["Max_Stock"]                     = item_class["Reorder_Point"] + item_class["Avg_Monthly"]

# Mask nhóm C
mask_cx_cy                                  = item_class['General_Class'].isin(['CX', 'CY'])
mask_cz                                     = item_class['General_Class'] == 'CZ'

# CX, CY — Min = 1 tháng, Max = 2 tháng
item_class.loc[mask_cx_cy, 'Safety_Stock']  = item_class.loc[mask_cx_cy, 'Avg_Monthly'].round(0)
item_class.loc[mask_cx_cy, 'Safety_Stock']  = (item_class.loc[mask_cx_cy, 'Avg_Monthly'] * 2).round(0)

# CZ — mua theo đơn
item_class.loc[mask_cz, 'Safety_Stock']     = 0
item_class.loc[mask_cz, 'Max_Stock']        = 0
item_class.head()

,Item Code,Supplier Code,Qty_PO,P90_LT,LT_P90_Supplier,LT_P90_Final,Count_Month_2025,Total_Qty_SO,Demand_Rate,Demand_Class,...,CV,XYZ,Avg_Daily,Std_Daily,ABC_2025,General_Class,Z_Score,Safety_Stock,Reorder_Point,Max_Stock
0,AAFK73C0-02,S00197,39,9.4000,7.0000,9.4000,10.0000,"1,965.0000",0.8333,Regular,...,1.0169,Z,5.4583,5.5504,C,CZ,NaN,0.0000,NaN,0.0000
1,MS-TP-8+8-M4-019,S00034,31,11.0000,13.0000,11.0000,11.0000,"51,542.0000",0.9167,Regular,...,0.7922,Y,143.1722,113.4282,A,AY,1.6500,620.7278,"2,195.6222","6,490.7889"
2,ALN-4-005,S00034,30,18.1000,13.0000,18.1000,12.0000,"830,000.0000",1.0000,Regular,...,0.2244,X,"2,305.5556",517.4427,C,CX,NaN,"138,333.0000",NaN,NaN
3,K1-K131-20-001,S00001,30,30.4000,20.8000,30.4000,12.0000,"55,810.0000",1.0000,Regular,...,0.4704,X,155.0278,72.9263,A,AX,1.6500,663.4452,"5,376.2897","10,027.1230"
4,8480150-3-005,S00034,30,18.1000,13.0000,18.1000,12.0000,"1,388,000.0000",1.0000,Regular,...,0.1948,X,"3,855.5556",750.9665,C,CX,NaN,"231,333.0000",NaN,NaN


### Module 5. Forecast Accuracy & Bias 

##### 1. Evaluate forecast by Same Month Last Year (SMLY)

In [37]:
# Filter closed SO delivered in 2024
so_2024_closed                         = df_so_sale_filtered[ (df_so["SO Status"]       == "Closed") &
                                                    (df_so["Delivery Year"]   == 2024)
                                                            ]

# Choose the AX, AY, BX, BY items in this list
item_class_2024                        = pd.merge(so_2024_closed, item_class[["Item Code", "General_Class" ]], on = "Item Code", how = "left")
item_class_2024                        = item_class_2024.dropna(subset=["General_Class"])

class_map                              = ["AX", "AY", "BX", "BY"]

item_general_2024                      = item_class_2024[item_class_2024["General_Class"].isin(class_map)]

# Create forecast 2025 propsed to be same as 2024
forecast_2025                          = item_general_2024.groupby(["Item Code", "Delivery Month"]).agg(FC_2025 = ('Delivery Qty','sum')).reset_index()
                                                
# Create actual delivery qty in 2025
actual_2025                            = so_closed_2025_item.groupby(["Item Code", "Delivery Month"]).agg(ACT_2025 = ("Delivery Qty" , 'sum')).reset_index()

# Compare forecast and actual 2025
compare_fc_so_2025                     = pd.merge(forecast_2025, actual_2025, on = ["Item Code", "Delivery Month"], how = "inner")

# Filtered item codes with active months. These are significantly reliable for MAPE calculation.
active_months                          = compare_fc_so_2025.groupby("Item Code")["Delivery Month"].count().reset_index()

active_items                           = active_months[active_months["Delivery Month"] >= 6]["Item Code"]
                                               
compare_active                        = compare_fc_so_2025[compare_fc_so_2025["Item Code"].isin(active_items)]

# MAPE and Bias
compare_active["ERROR"]               = compare_active["ACT_2025"] - compare_active["FC_2025"]
compare_active["APE"]                 = abs(compare_active["ERROR"]/compare_active["ACT_2025"])

compare_active_sum                    = compare_active.groupby("Item Code").agg(
                                                        MAPE = ("APE"   , "mean"),
                                                        BIAS = ("ERROR" , "mean")
                                                                ).reset_index()

compare_active_sum.head()



/var/folders/zn/76sp__ln5qg9wnm0lqlyfl800000gn/T/ipykernel_72697/1883977970.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  so_2024_closed                         = df_so_sale_filtered[ (df_so["SO Status"]       == "Closed") &


,Item Code,MAPE,BIAS
0,1100111-2+10-002,1.0487,"2,036.3636"
1,1100111-3+06-002,1.8323,171.2222
2,1100111-4+08-002,0.5816,"2,468.7500"
3,1100111-4+10-002,1.0021,537.5000
4,1200111-3+06-002,8.2216,-35.8333


#### 2. Filter the normal and abnormal-MAPE items to process the purchasing plan

The reliable-MAPE items: MAPE <= 30% to make the purchasing decisions

In [ ]:
compare_active_sum

SyntaxError: invalid syntax (2366041646.py, line 6)

Các bước tính Lead Time per item
Bước 1: Từ PO data 2025, tính lead time từng dòng
Lead Time = Receiving Plan Date − PO Date
Bước 2: Đếm số PO per Item + Supplier → tìm supplier có nhiều PO nhất per item → đó là primary supplier
Bước 3: Tính LT_P90 per Item + Primary Supplier

Item có ≥ 10 PO → dùng LT_P90 của item + primary supplier
Item có < 10 PO → dùng LT_P90 của primary supplier (không kể item)

Bước 4: Item nào không có PO trong 2025 → dùng LT_P90 theo Shipping Method phổ biến nhất của supplier đó

Bước 2: Phân loại demand (Regular / Seasonal / Sporadic)

Đếm số tháng có đơn / tổng tháng quan sát
Regular ≥ 80% → Seasonal 40–80% → Sporadic < 40%
Sporadic → dừng ở đây, xếp Z, mua theo đơn, không tính tiếp


Bước 3: Tính CV → phân loại XYZ

Regular: tính CV giữ tháng = 0
Seasonal: tính CV bỏ tháng = 0
Gán nhãn X / Y / Z theo ngưỡng CV


Bước 4: Tính ABC

Tính revenue = Delivery Qty × giá (từ sale_price)
Rank theo revenue tích lũy
A = top 80% revenue / B = 80–95% / C = 95–100%


Bước 5: Ghép ABC × XYZ → quyết định chiến lược
XYZACông thức đầy đủ, SL 98%Công thức, SL 95%Forecast theo projectBCông thức, SL 95%Công thức, SL 90%Safety stock cố địnhCMin/Max đơn giảnMin/Max đơn giảnMua theo đơn

Bước 6: Tính Lead Time per item

Từ PO data: tính LT_P90 per Item Code
Item nào ít PO (< 3 lần) → dùng LT_P90 của supplier thay thế


Bước 7: Tính Safety Stock & Reorder Point
Chỉ áp cho nhóm AX, AY, BX, BY — các nhóm còn lại xử lý riêng:

Bước 1: Tính Avg Monthly và σ Monthly từ pivot
Bước 2: Chuyển sang Daily
Bước 3: Gộp lại 1 bảng

Safety Stock = Z × σ_daily × √LT_P90


Reorder Point = (Avg_Daily × LT_P90) + Safety Stock


Min = Safety Stock


Max = Reorder Point + Avg_Monthly


Bước 8: Xử lý ngoại lệ

AZ: làm việc với khách hàng lấy forecast, không dùng công thức
CZ / Sporadic: mua theo đơn, không giữ tồn
Item có CV > 1.5: review thủ công trước khi dùng kết quả


Bước 9: Validation

So sánh ROP tính được với tồn kho hiện tại
Kiểm tra những item có Min/Max bất thường (quá lớn hoặc = 0)
Cho buyer review các item critical trước khi áp dụng

SO data ──→ Phân loại Regular/Seasonal/Sporadic
                    ↓
              Tính CV → XYZ
                    ↓
PO data ──→ Tính Lead Time P90
                    ↓
SO data ──→ Tính ABC (revenue)
                    ↓
            Ghép ABC × XYZ
                    ↓
         Chọn công thức phù hợp
                    ↓
      Tính SS → ROP → Min → Max
                    ↓
              Validation

Các bước tính Forecast Accuracy

Bước 1: Lọc item cần tính

Lấy danh sách item thuộc nhóm AX, AY, BX, BY từ abc_xyz


Bước 2: Tính Actual 2024 theo tháng × item

Filter SO Closed, năm 2024
Groupby Item Code + Month → sum Delivery Qty
Ra bảng: Item Code | Month | Actual_2024


Bước 3: Tạo Forecast 2025 bằng SMLY

Forecast 2025 = Actual 2024 cùng tháng
Đổi tên cột: Actual_2024 → Forecast_2025
Đổi năm trong Month: 2024 → 2025


Bước 4: Lấy Actual 2025

Dùng lại data SO 2025 đã có
Groupby Item Code + Month → sum Delivery Qty
Ra bảng: Item Code | Month | Actual_2025


Bước 5: Merge Forecast vs Actual

Merge Forecast_2025 với Actual_2025 theo Item Code + Month
Dùng inner join → chỉ giữ tháng có cả 2


Bước 6: Tính Forecast Error
Error     = Actual_2025 - Forecast_2025
Abs_Error = |Error|
APE       = |Error| / Actual_2025    ← per dòng

Bước 7: Tính MAPE và Bias per item
MAPE = mean(APE) × 100%      ← trung bình APE của item đó qua 12 tháng
Bias = mean(Error)            ← dương = under-forecast, âm = over-forecast

Bước 8: Đánh giá kết quả
MAPEĐánh giá< 10%Rất tốt10–20%Tốt20–50%Chấp nhận được> 50%Kém — cần xem lại

Các cột cần có trong file Excel
Thông tin item:

Item Code
ABC_XYZ
Supplier Code (primary)
LT_P90_Final

Thông tin demand:

FC_T4 (Actual tháng 4/2025)
Avg_Monthly
Demand_T4 (cột chọn)
Demand_Source (SMLY hay Avg Monthly)
MAPE
Bias

Thông tin tồn kho & planning:

Current_Stock (tồn kho hiện tại)
Safety_Stock
ROP
Min
Max

Kết quả tính toán:

Stock_After_T4
Order_Qty (trước NG)
NG_Rate
Order_Qty_Final (sau NG)

Action:

Action (Đặt ngay / Đặt ngay - Review MAPE / Chưa cần)

Các cột cần có trong file Excel
Thông tin item:

Item Code
ABC_XYZ
Supplier Code (primary)
LT_P90_Final

Thông tin demand:

FC_T4 (Actual tháng 4/2025)
Avg_Monthly
Demand_T4 (cột chọn)
Demand_Source (SMLY hay Avg Monthly)
MAPE
Bias

Thông tin tồn kho & planning:

Current_Stock (tồn kho hiện tại)
Safety_Stock
ROP
Min
Max

Kết quả tính toán:

Stock_After_T4
Order_Qty (trước NG)
NG_Rate
Order_Qty_Final (sau NG)

Action:

Action (Đặt ngay / Đặt ngay - Review MAPE / Chưa cần)